# 1/f noise fit diagnostic

Step-through of `NoiseStatsLevel2` for a single observation. Reproduces the
exact masking / interpolation / FFT path so you can see what data is being
fed to `fnoise_fit` and how the fit responds.

What you can inspect:
1. Raw TOD for a chosen (feed, band, channel, scan)
2. Bright-source mask (if a Planck 30 GHz map was supplied)
3. 3-sigma outlier mask from the median-filter residual
4. Final interpolated TOD that goes into the FFT
5. Power spectrum + log-binned points + best-fit model overlay
6. Sweep across channels in a band

All operations are read-only against the Level-2 HDF5 file.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

DB_PATH = os.path.join(REPO, 'databases', 'COMAP_manchester.db')

import numpy as np
import h5py
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import pandas as pd

from modules.ProcessLevel2.NoiseStats.NoiseStats import NoiseStatsLevel2
from modules.utils.median_filter import medfilt

## 1. Pick an observation

Set `OBSID` directly, or leave `None` to pick the first obsid with a local
Level-2 file from the SQL database.

In [ ]:
OBSID = None         # e.g. 11755
LEVEL2_PATH = None   # overrides the DB lookup if set

if LEVEL2_PATH is None:
    eng = create_engine(f'sqlite:///{DB_PATH}')
    q = "SELECT obsid, level2_path, source, source_group FROM comap_data WHERE level2_path IS NOT NULL"
    if OBSID is not None:
        q += f" AND obsid = {int(OBSID)}"
    q += " LIMIT 50"
    df = pd.read_sql(q, eng)
    df['exists_local'] = df['level2_path'].apply(lambda p: isinstance(p, str) and os.path.exists(p))
    local = df[df['exists_local']]
    if local.empty:
        print('No Level-2 files reachable from this host. Set LEVEL2_PATH manually.')
        print(df.head())
    else:
        row = local.iloc[0]
        OBSID = int(row['obsid'])
        LEVEL2_PATH = row['level2_path']

print('OBSID       =', OBSID)
print('LEVEL2_PATH =', LEVEL2_PATH)

In [ ]:
# Inspect what datasets are available and how many scans we have.
with h5py.File(LEVEL2_PATH, 'r') as f:
    feeds = f['spectrometer/feeds'][:]
    scan_edges = f['level2/scan_edges'][...]
    available = []
    for candidate in ['level2/binned_data', 'level2/binned_filtered_data']:
        if candidate in f:
            available.append((candidate, f[candidate].shape))
    central_freqs = f['level2/central_freqs'][...] if 'level2/central_freqs' in f else None

print('feeds       :', feeds.tolist())
print('n_scans     :', len(scan_edges))
print('scan_edges  :\n', scan_edges)
print('TOD datasets available:')
for name, shape in available:
    print(f'  {name}: shape={shape}  (nfeeds, nbands, n_channels, n_samples)')
if central_freqs is not None:
    print('central_freqs shape:', central_freqs.shape)

## 2. Choose what to look at

Pick a TOD dataset, feed, band, channel, and scan. These map directly onto
the indices used inside `NoiseStatsLevel2.fit_statistics`.

In [ ]:
TARGET_TOD_DATASET = 'level2/binned_filtered_data'  # or 'level2/binned_data'
FEED    = 1     # 1-indexed (skips GROUND_FEED automatically inside the module)
BAND    = 0
CHANNEL = 0
SCAN    = 0

# Optional: Planck 30 GHz map for bright-source masking. Leave None to skip.
PLANCK_30_PATH    = None
PLANCK_BRIGHT_CUT = 0.04

module = NoiseStatsLevel2(
    plot=False,
    target_tod_datasets=[TARGET_TOD_DATASET],
    planck_30_path=PLANCK_30_PATH,
    planck_bright_cut=PLANCK_BRIGHT_CUT,
)
module

## 3. Pull out the raw TOD and reproduce the masking pipeline

The cell below mirrors `NoiseStatsLevel2.fit_statistics` line-for-line for a
single (feed, band, channel, scan) and saves the intermediate arrays so we
can plot each stage.

In [ ]:
with h5py.File(LEVEL2_PATH, 'r') as f:
    scan_edges = f['level2/scan_edges'][...]
    scan_start, scan_end = scan_edges[SCAN]

    feeds = f['spectrometer/feeds'][:]
    ifeed = int(np.where(feeds == FEED)[0][0])

    raw = f[TARGET_TOD_DATASET][FEED-1, BAND, CHANNEL, scan_start:scan_end].astype(np.float64).copy()

    bright = module._build_bright_mask(f, ifeed, scan_start, scan_end)

auto_rms = module.auto_rms(raw)
print(f'scan {SCAN}: samples = {raw.size}')
print(f'auto_rms (differenced) = {auto_rms:.4g}')
print(f'NaN fraction in raw    = {np.mean(~np.isfinite(raw)):.3%}')

# Stage 1: bright-source interpolation (only if a Planck map was provided)
data = raw.copy()
stage1_skipped = True
if bright is not None and np.any(bright):
    stage1_skipped = False
    good = ~bright & np.isfinite(data)
    if good.sum() > 2:
        data[bright] = np.interp(
            np.where(bright)[0],
            np.where(good)[0],
            data[good],
        ) + np.random.normal(size=int(bright.sum()), scale=auto_rms)
    else:
        print('WARN: <=2 good samples after bright mask — module would skip this fit.')

# Stage 2: median-filtered residual + 3-sigma outlier rejection
data_filtered = data - np.array(medfilt.medfilt(data.copy(), 50))
fill_in = (np.abs(data_filtered) < 3*auto_rms)
good_frac = np.sum(fill_in) / data.size
print(f'fraction kept after 3sigma cut = {good_frac:.3%} (module needs >= 98%)')

# Stage 3: interpolate the rejected samples — this is what enters the FFT
data_for_fft = data.copy()
if good_frac >= 0.98 and np.any(~fill_in):
    data_for_fft[~fill_in] = np.interp(
        np.where(~fill_in)[0],
        np.where(fill_in)[0],
        data[fill_in],
    )

fit_result, k, Pk = module.fnoise_fit(data_for_fft.copy(), auto_rms, return_spectra=True)
sigma_white, sigma_red, alpha = fit_result
print(f'\nfit: sigma_white={sigma_white:.4g}  sigma_red={sigma_red:.4g}  alpha={alpha:.3f}')

## 4. Plot TOD at each masking stage

* **Top:** raw TOD with bright-source samples (if any) shaded.
* **Middle:** median-filter residual with the 3-sigma envelope; rejected
  samples are highlighted.
* **Bottom:** final TOD fed to the FFT (rejected samples interpolated).

In [ ]:
n = raw.size
t = np.arange(n)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

# --- raw + bright mask
axes[0].plot(t, raw, lw=0.6, label='raw TOD')
if bright is not None and np.any(bright):
    axes[0].fill_between(t, np.nanmin(raw), np.nanmax(raw),
                         where=bright, color='red', alpha=0.15,
                         label=f'bright mask ({bright.sum()} samp)')
axes[0].set_ylabel('raw TOD')
axes[0].set_title(f'obsid={OBSID}  feed={FEED}  band={BAND}  channel={CHANNEL}  scan={SCAN}')
axes[0].legend(loc='upper right', fontsize=8)

# --- median-filter residual + 3 sigma envelope
axes[1].plot(t, data_filtered, lw=0.6, label='data - medfilt(50)')
axes[1].axhline( 3*auto_rms, color='k', lw=0.7, ls='--', label='+/- 3 * auto_rms')
axes[1].axhline(-3*auto_rms, color='k', lw=0.7, ls='--')
rejected = ~fill_in
if np.any(rejected):
    axes[1].scatter(t[rejected], data_filtered[rejected], s=6, color='red',
                    label=f'rejected ({rejected.sum()} samp)')
axes[1].set_ylabel('residual')
axes[1].legend(loc='upper right', fontsize=8)

# --- final TOD passed to FFT
axes[2].plot(t, data_for_fft, lw=0.6, label='input to FFT')
if np.any(rejected):
    axes[2].scatter(t[rejected], data_for_fft[rejected], s=6, color='orange',
                    label='interpolated samples')
axes[2].set_ylabel('TOD into FFT')
axes[2].set_xlabel('sample index within scan')
axes[2].legend(loc='upper right', fontsize=8)

plt.tight_layout()

## 5. Plot the Fourier transform and fitted model

Two panels:
* **Left:** the raw periodogram (|FFT|^2 / N), plus the module's log-binned
  spectrum that is actually used in the fit chi^2.
* **Right:** binned spectrum with the best-fit `sigma_white^2 + sigma_red^2 (f/f_ref)^alpha`
  overlaid, and the white/red components shown separately.

In [ ]:
SAMPLE_FREQ = 50.0   # matches NoiseStatsLevel2.power_spectrum default
K_REF       = 0.1    # matches the module

# Full (un-binned) periodogram for context
x = data_for_fft - np.nanmedian(data_for_fft)
x[~np.isfinite(x)] = 0.0
N = x.size
raw_Pk = np.abs(np.fft.fft(x)[:N//2])**2 / N
raw_k  = np.fft.fftfreq(N, d=1./SAMPLE_FREQ)[:N//2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].loglog(raw_k[1:], raw_Pk[1:], color='0.7', lw=0.5, label='|FFT|^2 / N')
axes[0].loglog(k, Pk, 'o', ms=4, color='C0', label='log-binned (used in fit)')
axes[0].set_xlabel('frequency [Hz]')
axes[0].set_ylabel('power')
axes[0].set_title('periodogram')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.3)

axes[1].loglog(k, Pk, 'o', ms=4, color='C0', label='binned Pk')
if np.all(np.isfinite(fit_result)):
    model_full = module.model(k, sigma_white, sigma_red, alpha, k_ref=K_REF)
    axes[1].loglog(k, model_full, '-',  color='C3', lw=2,
                   label=f'model: a={alpha:.2f}')
    axes[1].loglog(k, np.full_like(k, sigma_white**2), '--', color='C2',
                   label=f'white = {sigma_white:.3g}')
    axes[1].loglog(k, sigma_red**2 * np.abs(k/K_REF)**alpha, ':', color='C1',
                   label=f'red   = {sigma_red:.3g}')
    # knee = where red == white
    if sigma_red > 0 and sigma_white > 0 and alpha < 0:
        f_knee = K_REF * (sigma_white/sigma_red)**(2.0/alpha)
        axes[1].axvline(f_knee, color='k', lw=0.8, ls='-.',
                        label=f'f_knee = {f_knee:.3g} Hz')
else:
    axes[1].text(0.5, 0.5, 'fit failed', transform=axes[1].transAxes,
                 ha='center', va='center', color='red')
axes[1].set_xlabel('frequency [Hz]')
axes[1].set_ylabel('power')
axes[1].set_title('fit overlay')
axes[1].legend(fontsize=8)
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()

## 6. Sweep across channels in a band

Quick overview to spot pathological channels: fits each channel of `BAND`
for the selected feed/scan and plots the binned power spectrum + best-fit
model overlay. Set `SWEEP_CHANNELS` to a small subset if `n_channels` is
large.

In [ ]:
SWEEP_CHANNELS = None  # None = all available channels

with h5py.File(LEVEL2_PATH, 'r') as f:
    n_channels_avail = f[TARGET_TOD_DATASET].shape[2]
    feed_band = f[TARGET_TOD_DATASET][FEED-1, BAND, :, scan_start:scan_end].astype(np.float64)

chans = list(range(n_channels_avail)) if SWEEP_CHANNELS is None else list(SWEEP_CHANNELS)

ncols = min(4, len(chans))
nrows = int(np.ceil(len(chans) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.5*nrows), squeeze=False)

for ax, ch in zip(axes.ravel(), chans):
    d = feed_band[ch].copy()
    if np.all(np.isnan(d)):
        ax.set_title(f'ch {ch}: all NaN'); ax.axis('off'); continue
    rms = module.auto_rms(d)
    df_resid = d - np.array(medfilt.medfilt(d.copy(), 50))
    keep = np.abs(df_resid) < 3*rms
    if keep.sum()/d.size < 0.98:
        ax.set_title(f'ch {ch}: kept={keep.sum()/d.size:.2%} <98%'); ax.axis('off'); continue
    d[~keep] = np.interp(np.where(~keep)[0], np.where(keep)[0], d[keep])
    (sw, sr, al), kk, Pkk = module.fnoise_fit(d, rms, return_spectra=True)
    ax.loglog(kk, Pkk, 'o', ms=3, color='C0')
    if np.all(np.isfinite([sw, sr, al])):
        ax.loglog(kk, module.model(kk, sw, sr, al), '-', color='C3', lw=1.5)
    ax.set_title(f'ch {ch}  a={al:.2f}\nw={sw:.2g} r={sr:.2g}', fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

for ax in axes.ravel()[len(chans):]:
    ax.axis('off')

plt.tight_layout()

## 7. Compare to stored fit results (sanity check)

If the module has already been run on this obsid, compare our recomputed
values to what is in `level2_noise_stats` inside the HDF5 file.

In [ ]:
stored_group = f'level2_noise_stats/{TARGET_TOD_DATASET.split("/")[-1]}'
with h5py.File(LEVEL2_PATH, 'r') as f:
    if stored_group in f:
        rec = {k: f[f'{stored_group}/{k}'][FEED-1, BAND, CHANNEL, SCAN] for k in f[stored_group].keys()}
    else:
        rec = None

print('Recomputed in this notebook:')
print(f'  sigma_white={sigma_white:.4g}  sigma_red={sigma_red:.4g}  alpha={alpha:.3f}  auto_rms={auto_rms:.4g}')
if rec is not None:
    print('\nStored in HDF5 ({}):'.format(stored_group))
    for k, v in rec.items():
        print(f'  {k} = {v:.4g}')
else:
    print(f'\n(no stored {stored_group} group — module has not been run on this file yet)')